# Phase 1 check — retrieval vs. gold citations

Runs **Phase 1 only** (PERCEIVE → FORMULATE → RETRIEVE ↔ ASSESS, via
`agent.pipeline.run_phase1`) for a handful of items and lets you inspect:

- PERCEIVE's output (what the model thinks it's looking at)
- each refine-loop iteration's query text, retrieved chunk ids, and ASSESS's
  `reason` (why it kept looping or stopped)
- the final retrieved chunk set vs. the item's **gold** chunk ids
  (`law` field in `law.json`, resolved through `CitationIndex`)

No SELECT call, no options, no answer — this notebook is purely about
retrieval quality and the ASSESS loop's reasoning, independent of Phase 2.

**Kernel**: run this under the `traffic` conda env kernel — it's the one
with pyserini + the bundled Java 21 JDK (needed for BM25) and the matching
`torch`/`torchvision` cu128 build (needed for `--llm vlm`) already set up.
If you only want to smoke-test the plumbing without a GPU, set
`LLM_BACKEND = "mock"` below and skip the VLM load entirely.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agent").exists():
    ROOT = ROOT.parent  # running from notebooks/, repo root is one level up
assert (ROOT / "agent").exists(), f"can't find the agent/ package from {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from PIL import Image

from agent.cli import resolve_image_path
from agent.corpus import Corpus
from agent.pipeline import run_phase1
from agent.resolve import CitationIndex
from agent.retrieval.bm25 import DEFAULT_INDEX_DIR, BM25Retriever
from agent.retrieval.mock import MockRetriever


## Config — edit this cell, then run everything below

In [ ]:
# --- data ---
DATA_PATH = ROOT / "law.json"
CORPUS_PATH = ROOT / "law_db.json"
DATA_ROOT = ROOT

# --- retriever: "bm25" (real, needs the Lucene index built once via
#     `python -m agent.build_bm25_index`) or "mock" (deterministic, no index) ---
RETRIEVER = "bm25"
INDEX_DIR = DEFAULT_INDEX_DIR
BM25_K1, BM25_B = 0.9, 0.4

# --- llm: "mock" (deterministic, zero-download, for plumbing checks) or
#     "vlm" (real model via answer_vqa.PRESETS — needs a GPU node) ---
LLM_BACKEND = "vlm"
MODEL_PRESET = "qwen3vl-2b"  # see answer_vqa.PRESETS for the full list
DEVICE = "cuda"
DTYPE = "bfloat16"

# --- phase 1 knobs (mirrors agent.cli defaults) ---
K = 3                  # chunks retrieved per query, per round
MAX_ITERATIONS = 2     # RETRIEVE rounds; ASSESS runs on all but the last
AS_OF = None           # effective_date filter, e.g. "2025-01-01"

# --- how many items to look at ---
N_SAMPLES = 20          # batch evaluation sample size
ITEM_INDEX = 0          # index into `records` for the single-item deep dive


In [ ]:
records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
corpus = Corpus.load(CORPUS_PATH)
citation_index = CitationIndex.from_file(CORPUS_PATH)
print(f"{len(records)} questions | corpus: {len(corpus)} chunks (v{corpus.version})")


In [ ]:
if RETRIEVER == "bm25":
    retriever = BM25Retriever(index_dir=INDEX_DIR, k1=BM25_K1, b=BM25_B)
else:
    retriever = MockRetriever(corpus)
print(f"retriever: {RETRIEVER}")


In [ ]:
if LLM_BACKEND == "vlm":
    from agent.llm_vlm import VLMLLMClient  # heavy: pulls in torch/transformers
    import answer_vqa

    preset = answer_vqa.PRESETS[MODEL_PRESET]
    llm = VLMLLMClient(preset.hf_id, preset.family, device=DEVICE, dtype=DTYPE)
    model_label = MODEL_PRESET
else:
    from agent.llm import MockLLM

    llm = MockLLM()
    model_label = "mock"
print(f"llm: {LLM_BACKEND} ({model_label})")


## Helper: run Phase 1 on one item and score retrieval against gold

In [ ]:
def run_and_compare(rec: dict) -> dict:
    image_path = resolve_image_path(rec["image"], DATA_ROOT)
    gold_resolved = citation_index.resolve_set(rec.get("law", []))

    perception, retrieved, trace = run_phase1(
        image_path, rec["question"], corpus, retriever, llm,
        k=K, as_of=AS_OF, max_iterations=MAX_ITERATIONS,
    )
    retrieved_ids = {c.chunk_id for c in retrieved}
    overlap = retrieved_ids & gold_resolved

    n_gold = len(gold_resolved)
    recall = len(overlap) / n_gold if n_gold else None
    precision = len(overlap) / len(retrieved_ids) if retrieved_ids else None

    return {
        "item_id": rec.get("id"),
        "question": rec["question"],
        "image_path": image_path,
        "type_image": rec.get("type_image"),
        "perception": perception,
        "trace": trace,
        "retrieved": retrieved,
        "retrieved_ids": retrieved_ids,
        "gold_ids": gold_resolved,
        "overlap_ids": overlap,
        "n_gold": n_gold,
        "n_retrieved": len(retrieved_ids),
        "n_overlap": len(overlap),
        "recall": recall,
        "precision": precision,
        "iterations_used": trace.iterations_used,
        "stop_reason": trace.stop_reason,
    }


def print_result(res: dict) -> None:
    print(f"item_id={res['item_id']}  type_image={res['type_image']}")
    print(f"Q: {res['question']}\n")

    p = res["perception"]
    print("PERCEIVE:")
    print(f"  focus:     {p.focus}")
    print(f"  vehicle:   {p.vehicle}")
    print(f"  observed:  {p.observed}")
    print(f"  violation: {p.violation}")
    print(f"  confidence:{p.confidence}\n")

    print(f"ASSESS loop ({res['iterations_used']} iteration(s), stop_reason={res['stop_reason']}):")
    for it in res["trace"].iterations:
        print(f"  iter {it.iteration}: query={it.query_texts}")
        print(f"    retrieved this round: {it.retrieved_chunk_ids}")
        if it.assessed:
            print(f"    sufficient={it.sufficient}  refined_query={it.refined_query!r}")
            print(f"    reason: {it.reason}")
        else:
            print("    (assess skipped — final allowed round)")
    print()

    print(f"retrieved ({res['n_retrieved']}): {sorted(res['retrieved_ids'])}")
    print(f"gold      ({res['n_gold']}): {sorted(res['gold_ids'])}")
    print(f"overlap   ({res['n_overlap']}): {sorted(res['overlap_ids'])}")
    print(f"recall={res['recall']}  precision={res['precision']}\n")

    missed = res["gold_ids"] - res["retrieved_ids"]
    if missed:
        print("MISSED gold chunks (in gold, never retrieved):")
        for cid in sorted(missed):
            chunk = corpus.by_id.get(cid)
            text = chunk.text[:300] + ("…" if chunk and len(chunk.text) > 300 else "") if chunk else "(not in corpus!)"
            print(f"  [{cid}] {text}")

    extra = res["retrieved_ids"] - res["gold_ids"]
    if extra:
        print("\nEXTRA retrieved chunks (retrieved, not gold):")
        for cid in sorted(extra):
            chunk = corpus.by_id.get(cid)
            text = chunk.text[:300] + ("…" if chunk and len(chunk.text) > 300 else "") if chunk else "(not in corpus!)"
            print(f"  [{cid}] {text}")


## Single-item deep dive

In [ ]:
rec = records[ITEM_INDEX]
result = run_and_compare(rec)

display(Image.open(result["image_path"]))
options = {L: rec[L] for L in ("A", "B", "C", "D")}
for L, text in options.items():
    marker = " <- GT" if rec.get("GT") == L else ""
    print(f"{L}. {text}{marker}")
print()
print_result(result)


## Batch evaluation

Runs Phase 1 over the first `N_SAMPLES` items and summarizes retrieval
recall/precision against gold citations. This is the model+retriever combo
selected in the Config cell above.


In [ ]:
rows = []
results_by_item = {}
for rec in records[:N_SAMPLES]:
    res = run_and_compare(rec)
    results_by_item[res["item_id"]] = res
    rows.append({
        "item_id": res["item_id"],
        "type_image": res["type_image"],
        "n_gold": res["n_gold"],
        "n_retrieved": res["n_retrieved"],
        "n_overlap": res["n_overlap"],
        "recall": res["recall"],
        "precision": res["precision"],
        "iterations_used": res["iterations_used"],
        "stop_reason": res["stop_reason"],
    })

df = pd.DataFrame(rows)
df


In [ ]:
graded = df.dropna(subset=["recall"])
print(f"{len(graded)}/{len(df)} items have at least one gold citation")
print(f"mean recall:    {graded['recall'].mean():.3f}")
print(f"mean precision: {graded['precision'].dropna().mean():.3f}")
print(f"full recall (all gold chunks retrieved): {(graded['recall'] == 1.0).sum()}/{len(graded)}")
print(f"zero recall  (no gold chunk retrieved):  {(graded['recall'] == 0.0).sum()}/{len(graded)}")
print()
print("stop_reason counts:")
print(df["stop_reason"].value_counts())


## Worst-recall items — inspect where retrieval missed gold entirely

In [ ]:
worst = graded.sort_values("recall").head(10)
worst


In [ ]:
# Deep-dive a specific worst item by item_id from the table above.
WORST_ITEM_ID = worst.iloc[0]["item_id"] if len(worst) else None
if WORST_ITEM_ID is not None:
    print_result(results_by_item[WORST_ITEM_ID])
